# Exploración de PeruFoodNet

El objetivo es reconocer platos peruanos a partir de fotografías. Una clase es el nombre del plato asignado por la carpeta. Exploraremos los datos antes de entrenar. PeruFoodNet v3 se distribuye bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/); cite la [fuente](https://data.mendeley.com/datasets/hxhbbm497d/3).

In [ ]:
from pathlib import Path
import sys
DATASET_URL = "https://data.mendeley.com/public-api/zip/hxhbbm497d/download/3"
DATASET_ROOT = Path(r"E:/Dataset PeruFoodNet")
PROJECT_OVERRIDE = None  # En Colab, indique aquí la ruta del proyecto si no está en el directorio actual.
DOWNLOAD_DIR = Path("perufoodnet_data")
current = Path.cwd()
candidates = [current, current.parent, current / "examen-final/perufoodnet"]
PROJECT = Path(PROJECT_OVERRIDE) if PROJECT_OVERRIDE else next(
    (path for path in candidates if (path / "data/manifest.csv").is_file()), None
)
if PROJECT is None:
    raise FileNotFoundError("Configure PROJECT_OVERRIDE con la ruta del proyecto")
MANIFEST = PROJECT / "data/manifest.csv"
CLASSES = PROJECT / "data/classes.json"
sys.path.insert(0, str(PROJECT / "src"))

## Recursos
Se usa primero la copia local. Si falta, el programa descarga el ZIP desde el enlace configurado y busca su carpeta interna `Dataset PeruFoodNet`. Acepta enlaces HTTP directos o públicos de Google Drive. En Colab indique la ruta del proyecto en `PROJECT_OVERRIDE` si es necesario.

In [ ]:
from resources import resolve_dataset
DATASET_ROOT = resolve_dataset(DATASET_ROOT, DATASET_URL, DOWNLOAD_DIR)
print("Datos:", DATASET_ROOT, "Manifiesto:", MANIFEST)

## Etiquetas y distribución
El manifiesto asocia cada ruta con un índice de clase. Los conteos esperados son 100 imágenes por cada una de las 40 clases.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
rows = pd.read_csv(MANIFEST)
classes = json.loads(CLASSES.read_text(encoding="utf-8-sig"))
print("Imágenes:", len(rows), "Clases:", len(classes))
display(rows.head().rename(columns={"relative_path": "ruta", "class_name": "clase", "class_index": "índice", "image_number": "número", "split": "partición"}))
display(rows.groupby("class_name").size().rename("imágenes").to_frame())

## Ejemplos
Cada imagen usa la etiqueta de su carpeta. Observe diferencias de presentación, iluminación y encuadre; no son una auditoría exhaustiva de las etiquetas.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, (_, row) in zip(axes.flat, rows.groupby("class_name").first().reset_index().head(8).iterrows()):
    with Image.open(DATASET_ROOT / row.relative_path) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(row.class_name); ax.axis("off")
plt.tight_layout()

## Tamaño y calidad básica
Las dimensiones originales pueden variar; el modelo las ajustará a 224 × 224. La comprobación identifica imágenes que PIL no puede abrir, no errores semánticos ni duplicados.

In [ ]:
sizes, errors = [], []
for path in rows.relative_path:
    try:
        with Image.open(DATASET_ROOT / path) as image:
            sizes.append(image.size)
            image.verify()
    except Exception as exc:
        errors.append((path, str(exc)))
print("Errores de lectura:", len(errors), errors[:5])
display(pd.DataFrame(sizes, columns=["ancho", "alto"]).describe())

## Partición secuencial
Por clase, 01–70 se reserva para entrenamiento, 71–85 para prueba y 86–100 para validación. Las pruebas usan 01–10 o 01–40 de entrenamiento, por lo que el conjunto de 10 está incluido en el de 40. La validación permite comparar configuraciones; la prueba se reserva para el ganador.

In [ ]:
display(pd.crosstab(rows.class_name, rows.split).rename_axis("clase").rename(columns={"train": "entrenamiento", "val": "validación", "test": "prueba"}))
print("Subconjuntos de entrenamiento:", int(rows.subset_10.sum()), int(rows.subset_40.sum()))
print("Anidamiento:", bool((rows.subset_10 <= rows.subset_40).all()))

## Interpretación y límites
La división separa archivos, pero tomas de una misma ración podrían aparecer en distintas particiones. Esa correlación puede elevar las métricas; las conclusiones se limitan a esta partición. Este cuaderno no entrena ni evalúa modelos.